# Assignment: Real-Time Intrusion Detection with Scapy + PyTorch

## Task 1: Dataset Exploration and Feature Understanding


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

DATA_PATH = r'd:\Dev\MDLS\Datasett'

files = glob.glob(os.path.join(DATA_PATH, '*.csv'))
# Exclude generated logs
files = [f for f in files if 'network_log.csv' not in os.path.basename(f)]

print(f"Found {len(files)} files.")

if files:
    # LOAD ALL DATA
    print("Loading ALL files... this may take memory.")
    li = []
    for filename in files:
        df_temp = pd.read_csv(filename, index_col=None, header=0)
        li.append(df_temp)
    
    df = pd.concat(li, axis=0, ignore_index=True)
    
    # Strip whitespace from headers
    df.columns = df.columns.str.strip()
    
    print(f"Full Dataset Shape: {df.shape}")
else:
    print("No data found.")


### 1. List of All Features
Loading the dataset and listing all available feature names.


In [ ]:
if 'df' in locals():
    print(f"Total Features: {len(df.columns)}")
    display(pd.DataFrame(df.columns, columns=['Feature Name']))


### 2. Feature Analysis (Name, Type, Description, Relevance)
The following table provides the Name, Data Type, Description, and Justification (Relevance) for **each** feature.


In [ ]:
# Comprehensive Feature Definitions
features_metadata = {
    'Destination Port': {'type': 'int', 'desc': 'Destination port number', 'level': 'Packet', 'relevance': 'Identifies compromised service.'},
    'Flow Duration': {'type': 'int', 'desc': 'Duration of the flow in microseconds', 'level': 'Flow', 'relevance': 'DDoS/Bruteforce result in short/long durations.'},
    'Total Fwd Packets': {'type': 'int', 'desc': 'Total packets in forward direction', 'level': 'Flow', 'relevance': 'Volume metric.'},
    'Total Backward Packets': {'type': 'int', 'desc': 'Total packets in backward direction', 'level': 'Flow', 'relevance': 'Volume metric.'},
    'Total Length of Fwd Packets': {'type': 'int', 'desc': 'Total size of packet payload in forward direction', 'level': 'Flow', 'relevance': 'Data exfiltration volume.'},
    'Total Length of Bwd Packets': {'type': 'int', 'desc': 'Total size of packet payload in backward direction', 'level': 'Flow', 'relevance': 'Download volume.'},
    'Fwd Packet Length Max': {'type': 'int', 'desc': 'Maximum size of packet in forward direction', 'level': 'Flow', 'relevance': 'MTU anomalies.'},
    'Fwd Packet Length Min': {'type': 'int', 'desc': 'Minimum size of packet in forward direction', 'level': 'Flow', 'relevance': 'Control packets.'},
    'Fwd Packet Length Mean': {'type': 'float', 'desc': 'Average size of packet in forward direction', 'level': 'Flow', 'relevance': 'Profile of application protocol.'},
    'Fwd Packet Length Std': {'type': 'float', 'desc': 'Standard deviation size of packet in forward direction', 'level': 'Flow', 'relevance': 'Variability of payload.'},
    'Bwd Packet Length Max': {'type': 'int', 'desc': 'Maximum size of packet in backward direction', 'level': 'Flow', 'relevance': ''},
    'Bwd Packet Length Min': {'type': 'int', 'desc': 'Minimum size of packet in backward direction', 'level': 'Flow', 'relevance': ''},
    'Bwd Packet Length Mean': {'type': 'float', 'desc': 'Mean size of packet in backward direction', 'level': 'Flow', 'relevance': ''},
    'Bwd Packet Length Std': {'type': 'float', 'desc': 'Standard deviation size of packet in backward direction', 'level': 'Flow', 'relevance': ''},
    'Flow Bytes/s': {'type': 'float', 'desc': 'Flow throughput in bytes per second', 'level': 'Flow', 'relevance': 'Bandwidth usage.'},
    'Flow Packets/s': {'type': 'float', 'desc': 'Flow throughput in packets per second', 'level': 'Flow', 'relevance': 'High PPS indicates DDoS.'},
    'Flow IAT Mean': {'type': 'float', 'desc': 'Mean time between two packets in the flow', 'level': 'Flow', 'relevance': 'Automated traffic has regular IAT.'},
    'Flow IAT Std': {'type': 'float', 'desc': 'Standard deviation of time between two packets', 'level': 'Flow', 'relevance': 'Jitter.'},
    'Flow IAT Max': {'type': 'int', 'desc': 'Maximum time between two packets', 'level': 'Flow', 'relevance': 'Idle periods.'},
    'Flow IAT Min': {'type': 'int', 'desc': 'Minimum time between two packets', 'level': 'Flow', 'relevance': 'Bursts.'},
    'Fwd IAT Total': {'type': 'int', 'desc': 'Total time between two forward packets', 'level': 'Flow', 'relevance': ''},
    'Fwd IAT Mean': {'type': 'float', 'desc': 'Mean time between two forward packets', 'level': 'Flow', 'relevance': ''},
    'Fwd IAT Std': {'type': 'float', 'desc': 'Standard deviation of time between two forward packets', 'level': 'Flow', 'relevance': ''},
    'Fwd IAT Max': {'type': 'int', 'desc': 'Maximum time between two forward packets', 'level': 'Flow', 'relevance': ''},
    'Fwd IAT Min': {'type': 'int', 'desc': 'Minimum time between two forward packets', 'level': 'Flow', 'relevance': ''},
    'Bwd IAT Total': {'type': 'int', 'desc': 'Total time between two backward packets', 'level': 'Flow', 'relevance': ''},
    'Bwd IAT Mean': {'type': 'float', 'desc': 'Mean time between two backward packets', 'level': 'Flow', 'relevance': ''},
    'Bwd IAT Std': {'type': 'float', 'desc': 'Standard deviation of time between two backward packets', 'level': 'Flow', 'relevance': ''},
    'Bwd IAT Max': {'type': 'int', 'desc': 'Maximum time between two backward packets', 'level': 'Flow', 'relevance': ''},
    'Bwd IAT Min': {'type': 'int', 'desc': 'Minimum time between two backward packets', 'level': 'Flow', 'relevance': ''},
    'Fwd PSH Flags': {'type': 'int', 'desc': 'Number of times the PSH flag was set in forward direction', 'level': 'Flow/Packet', 'relevance': 'Data push urgency.'},
    'Bwd PSH Flags': {'type': 'int', 'desc': 'Number of times the PSH flag was set in backward direction', 'level': 'Flow/Packet', 'relevance': ''},
    'Fwd URG Flags': {'type': 'int', 'desc': 'Number of times the URG flag was set in forward direction', 'level': 'Flow/Packet', 'relevance': ''},
    'Bwd URG Flags': {'type': 'int', 'desc': 'Number of times the URG flag was set in backward direction', 'level': 'Flow/Packet', 'relevance': ''},
    'Fwd Header Length': {'type': 'int', 'desc': 'Total bytes used for headers in forward direction', 'level': 'Flow', 'relevance': 'Tunneling overhead.'},
    'Bwd Header Length': {'type': 'int', 'desc': 'Total bytes used for headers in backward direction', 'level': 'Flow', 'relevance': ''},
    'Fwd Packets/s': {'type': 'float', 'desc': 'Number of forward packets per second', 'level': 'Flow', 'relevance': ''},
    'Bwd Packets/s': {'type': 'float', 'desc': 'Number of backward packets per second', 'level': 'Flow', 'relevance': ''},
    'Min Packet Length': {'type': 'int', 'desc': 'Minimum length of a packet', 'level': 'Flow', 'relevance': ''},
    'Max Packet Length': {'type': 'int', 'desc': 'Maximum length of a packet', 'level': 'Flow', 'relevance': ''},
    'Packet Length Mean': {'type': 'float', 'desc': 'Mean length of a packet', 'level': 'Flow', 'relevance': ''},
    'Packet Length Std': {'type': 'float', 'desc': 'Standard deviation of length of a packet', 'level': 'Flow', 'relevance': ''},
    'Packet Length Variance': {'type': 'float', 'desc': 'Variance of length of a packet', 'level': 'Flow', 'relevance': ''},
    'FIN Flag Count': {'type': 'int', 'desc': 'Number of packets with FIN', 'level': 'Flow', 'relevance': 'Connection termination.'},
    'SYN Flag Count': {'type': 'int', 'desc': 'Number of packets with SYN', 'level': 'Flow', 'relevance': 'Connection establishment.'},
    'RST Flag Count': {'type': 'int', 'desc': 'Number of packets with RST', 'level': 'Flow', 'relevance': 'Connection reset.'},
    'PSH Flag Count': {'type': 'int', 'desc': 'Number of packets with PSH', 'level': 'Flow', 'relevance': ''},
    'ACK Flag Count': {'type': 'int', 'desc': 'Number of packets with ACK', 'level': 'Flow', 'relevance': ''},
    'URG Flag Count': {'type': 'int', 'desc': 'Number of packets with URG', 'level': 'Flow', 'relevance': ''},
    'CWE Flag Count': {'type': 'int', 'desc': 'Number of packets with CWE', 'level': 'Flow', 'relevance': 'Congestion Window ECN.'},
    'ECE Flag Count': {'type': 'int', 'desc': 'Number of packets with ECE', 'level': 'Flow', 'relevance': 'ECN-Echo.'},
    'Down/Up Ratio': {'type': 'float', 'desc': 'Download and upload ratio', 'level': 'Flow', 'relevance': 'Traffic directionality.'},
    'Average Packet Size': {'type': 'float', 'desc': 'Average size of packet', 'level': 'Flow', 'relevance': ''},
    'Avg Fwd Segment Size': {'type': 'float', 'desc': 'Average size observed in the forward direction', 'level': 'Flow', 'relevance': ''},
    'Avg Bwd Segment Size': {'type': 'float', 'desc': 'Average size observed in the backward direction', 'level': 'Flow', 'relevance': ''},
    'Fwd Header Length.1': {'type': 'int', 'desc': 'Duplicate column (Header Length)', 'level': 'Flow', 'relevance': ''},
    'Fwd Avg Bytes/Bulk': {'type': 'int', 'desc': 'Average bytes/bulk in forward', 'level': 'Flow', 'relevance': ''},
    'Fwd Avg Packets/Bulk': {'type': 'int', 'desc': 'Average packets/bulk in forward', 'level': 'Flow', 'relevance': ''},
    'Fwd Avg Bulk Rate': {'type': 'int', 'desc': 'Average bulk rate in forward', 'level': 'Flow', 'relevance': ''},
    'Bwd Avg Bytes/Bulk': {'type': 'int', 'desc': 'Average bytes/bulk in backward', 'level': 'Flow', 'relevance': ''},
    'Bwd Avg Packets/Bulk': {'type': 'int', 'desc': 'Average packets/bulk in backward', 'level': 'Flow', 'relevance': ''},
    'Bwd Avg Bulk Rate': {'type': 'int', 'desc': 'Average bulk rate in backward', 'level': 'Flow', 'relevance': ''},
    'Subflow Fwd Packets': {'type': 'int', 'desc': 'The average number of packets in a subflow in the forward direction', 'level': 'Flow', 'relevance': ''},
    'Subflow Fwd Bytes': {'type': 'int', 'desc': 'The average number of bytes in a subflow in the forward direction', 'level': 'Flow', 'relevance': ''},
    'Subflow Bwd Packets': {'type': 'int', 'desc': 'The average number of packets in a subflow in the backward direction', 'level': 'Flow', 'relevance': ''},
    'Subflow Bwd Bytes': {'type': 'int', 'desc': 'The average number of bytes in a subflow in the backward direction', 'level': 'Flow', 'relevance': ''},
    'Init_Win_bytes_forward': {'type': 'int', 'desc': 'Total number of bytes sent in initial window in the forward direction', 'level': 'Flow', 'relevance': 'TCP Window scaling detected.'},
    'Init_Win_bytes_backward': {'type': 'int', 'desc': 'Total number of bytes sent in initial window in the backward direction', 'level': 'Flow', 'relevance': ''},
    'act_data_pkt_fwd': {'type': 'int', 'desc': 'Count of packets with at least 1 byte of TCP data payload in forward direction', 'level': 'Flow', 'relevance': 'Actual data transfer vs control.'},
    'min_seg_size_forward': {'type': 'int', 'desc': 'Minimum segment size observed in the forward direction', 'level': 'Flow', 'relevance': ''},
    'Active Mean': {'type': 'float', 'desc': 'Mean time a flow was active before becoming idle', 'level': 'Flow', 'relevance': 'Active/Sleep cycles.'},
    'Active Std': {'type': 'float', 'desc': 'Standard deviation time a flow was active', 'level': 'Flow', 'relevance': ''},
    'Active Max': {'type': 'int', 'desc': 'Maximum time a flow was active', 'level': 'Flow', 'relevance': ''},
    'Active Min': {'type': 'int', 'desc': 'Minimum time a flow was active', 'level': 'Flow', 'relevance': ''},
    'Idle Mean': {'type': 'float', 'desc': 'Mean time a flow was idle', 'level': 'Flow', 'relevance': 'Beaconing behavior.'},
    'Idle Std': {'type': 'float', 'desc': 'Standard deviation time a flow was idle', 'level': 'Flow', 'relevance': ''},
    'Idle Max': {'type': 'int', 'desc': 'Maximum time a flow was idle', 'level': 'Flow', 'relevance': ''},
    'Idle Min': {'type': 'int', 'desc': 'Minimum time a flow was idle', 'level': 'Flow', 'relevance': ''},
    'Label': {'type': 'str', 'desc': 'The class label (Benign or specific Attack)', 'level': 'Meta', 'relevance': 'Target variable.'}
}

def get_feature_info(col_name):
    # Try exact match or match derived names
    meta = features_metadata.get(col_name)
    if not meta:
        return {'Name': col_name, 'Type': 'Unknown', 'Description': 'Feature not manually mapped', 'Level': 'Unknown', 'Relevance': 'Unknown'}
    return {
        'Name': col_name,
        'Type': meta['type'],
        'Description': meta['desc'],
        'Level': meta['level'],
        'Relevance': meta['relevance']
    }

if 'df' in locals():
    feature_list_data = [get_feature_info(col) for col in df.columns]
    df_features = pd.DataFrame(feature_list_data)
    display(df_features)


### 3. Summary of Packet vs Flow Level Features

**Packet-Level Features** are those that can be derived from the headers or payload of a **single** packet, without any historical context or state.
*   **Examples**: `Destination Port` (TCP Header), `Protocol` (IP Header), `Packet Length` (IP Header), `Flags` (TCP Header) *of a specific packet*.
*   **Characteristics**: Extremely fast to extract (O(1)).

**Flow-Level (Derived/Aggregated) Features** are calculated by tracking statistics over a sequence of packets.
*   **Clarification on 'Packet Length Max'**: Even though `Fwd Packet Length Max` has "Packet" in the name, it is a **Flow** feature. Why? Because to know the "Max", you must observe *all* packets in the flow (or a window) to find the largest one. A single packet cannot know if it is the "Max" of the whole stream.
*   **Examples**: `Flow Duration`, `Total Fwd Packets`, `Fwd Packet Length Max/Mean/Std`.
*   **Characteristics**: Require state maintenance (hash tables) and memory.


## Task 2: Identify Scapy-Computable Features

We select a subset of features that are feasible to calculate in real-time using Scapy.

### Computability Table

| Feature | Linux Server? | Raspberry Pi? | Reason | Validation Snippet |
|---|---|---|---|---|
| **Src/Dst IP & Port** | Yes | Yes | Basic header parsing, negligible cost. | `pkt[IP].src`, `pkt[TCP].sport` |
| **Packet Length** | Yes | Yes | `len(pkt)`, immediate availability. | `len(packet)` |
| **TCP Flags** | Yes | Yes | Parsing TCP header is lightweight. | `pkt[TCP].flags` |
| **Flow Duration** | Yes | Yes (Low traffic) | Requires hash map lookup. On Pi, high RAM usage if many concurrent flows. | `now - flow_start_time[flow_id]` |
| **Packet Number/Count** | Yes | Yes (Low traffic) | Simple counter in hash map. | `flows[id]['count'] += 1` |
| **Inter-Arrival Time (IAT)** | Yes | Maybe | Requires storing timestamp of *last* packet. Calculation is cheap, but state management adds up. | `now - last_seen[flow_id]` |
| **Bytes/s** | Yes | Maybe | Derived from Count/Duration. Floating point div is cheap, but state is the bottleneck. | `total_bytes / duration` |

### Justification:
*   **Raspberry Pi Limitations**: Limited RAM and CPU. Keeping state for 100k+ concurrent flows (common in DDoS) will exhaust RAM or cause packet drops due to processing latency (Python/Scapy overhead). Feature extraction must be kept to O(1) or O(log n) with small n.
*   **Selected Subset for Task 3**:
    1.  **Basic 5-tuple**: Src IP, Dst IP, Src Port, Dst Port, Protocol.
    2.  **Packet Size**: `len(packet)`.
    3.  **TCP Flags**: (SYN, ACK, FIN, RST, PSH, URG).
    4.  **Flow Stats (Simplified)**: Packet Count, Byte Count (per flow).


## Task 3: Real-Time Scapy Extractor

This script will:
1.  Detect OS.
2.  Sniff traffic.
3.  Extract features.
4.  Log to CSV.


In [ ]:
import sys
import platform
import os
import time
import csv
from collections import defaultdict
from datetime import datetime
from scapy.all import sniff, IP, TCP, UDP, Ether, wrpcap

# 1. Detect OS
def get_system_info():
    sys_type = platform.system()
    node = platform.node()
    machine = platform.machine()
    is_pi = 'arm' in machine.lower() or 'aarch64' in machine.lower() or 'raspberry' in node.lower()
    return {
        "OS": sys_type,
        "Node": node,
        "Is_Pi": is_pi
    }

sys_info = get_system_info()
print(f"Running on: {sys_info['OS']} ({sys_info['Node']})")
print(f"Target Hardware: {'Raspberry Pi' if sys_info['Is_Pi'] else 'Likely Server/PC'}")

# 2. & 3. Scapy Extractor
class RealTimeExtractor:
    def __init__(self, log_file="network_log.csv"):
        self.log_file = log_file
        self.flow_table = defaultdict(lambda: {'start_time': 0, 'packet_count': 0, 'byte_count': 0})
        
        # Initialize CSV
        self.features = ['timestamp', 'src_ip', 'dst_ip', 'src_port', 'dst_port', 'proto', 
                         'flag', 'packet_len', 'flow_duration', 'flow_pkt_count']
        
        with open(self.log_file, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=self.features)
            writer.writeheader()
            
    def process_packet(self, pkt):
        try:
            if IP not in pkt:
                return

            row = {}
            ip = pkt[IP]
            row['timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')
            row['src_ip'] = ip.src
            row['dst_ip'] = ip.dst
            row['proto'] = ip.proto
            row['packet_len'] = len(pkt)
            
            # Layer 4
            sport = 0
            dport = 0
            flags = 'None'
            
            if TCP in pkt:
                sport = pkt[TCP].sport
                dport = pkt[TCP].dport
                flags = str(pkt[TCP].flags)
            elif UDP in pkt:
                sport = pkt[UDP].sport
                dport = pkt[UDP].dport
                flags = 'UDP'
            
            row['src_port'] = sport
            row['dst_port'] = dport
            row['flag'] = flags
            
            # Flow Tracking (Simplified)
            flow_id = (ip.src, ip.dst, sport, dport, ip.proto)
            curr_time = time.time()
            
            if self.flow_table[flow_id]['packet_count'] == 0:
                self.flow_table[flow_id]['start_time'] = curr_time
            
            self.flow_table[flow_id]['packet_count'] += 1
            self.flow_table[flow_id]['byte_count'] += len(pkt)
            
            row['flow_duration'] = curr_time - self.flow_table[flow_id]['start_time']
            row['flow_pkt_count'] = self.flow_table[flow_id]['packet_count']
            
            # Log
            with open(self.log_file, 'a', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=self.features)
                writer.writerow(row)
                
            # Verify/Print (Optional, print every 10th packet to reduce noise)
            if self.flow_table[flow_id]['packet_count'] % 10 == 1:
                print(f"Logged: {row['src_ip']} -> {row['dst_ip']} | Len: {row['packet_len']}")

        except Exception as e:
            # print(f"Error: {e}")
            pass

# Run Sniffer (Limited count for Notebook demonstration)
print("Starting Sniffer for 50 packets...")
extractor = RealTimeExtractor()

# We set a timeout and count so the notebook cell doesn't run forever
try:
    sniff(prn=extractor.process_packet, count=50, timeout=10, store=0)
    print("Sniffing completed. Check network_log.csv")
except Exception as e:
    print(f"Sniffing error (permissions?): {e}")


## Task 4: Prepare Platform-Specific Datasets

We will now prepare two sub-datasets using the **ENTIRE** dataset:
1.  **Raspberry Pi Dataset (`df_pi`)**: Includes only features computable in real-time on limited hardware (Packet-level + simplified Flow Stats).
2.  **Linux Server Dataset (`df_linux`)**: Includes all features (assuming high-performance hardware).

### Preprocessing Steps
1.  **Handling Infinity/NaN**: Replace logical infinite values with NaN, then drop rows containing NaN.
2.  **Encoding**: Convert string labels (`BENIGN`, `DDoS`, etc.) to integers using `LabelEncoder`.
3.  **Normalization**: Scale numeric features to mean=0, std=1 using `StandardScaler`.


In [ ]:
# Define Columns for Pi (Subset identified in Task 2)
columns_pi = [
    'Destination Port', 'Protocol',
    'Total Fwd Packets', 'Total Backward Packets', 
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
    'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean',
    'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count',
    'Down/Up Ratio', 'Label'
]

# 1. Use Data (Already loaded FULL df in Step 1)
if 'df' not in locals():
    print("DataFrame 'df' not found. Please run the Load Data cell first.")
else:
    print(f"Using full dataset with shape: {df.shape}")

    # 2. Split
    # Check if columns exist (ignoring missing ones for safety)
    cols_pi_present = [c for c in columns_pi if c in df.columns]
    df_pi = df[cols_pi_present].copy()
    df_linux = df.copy()

    print(f"Pi Dataset Shape: {df_pi.shape}")
    print(f"Linux Dataset Shape: {df_linux.shape}")

    # 3. Preprocessing Function
    def preprocess_dataset(dataframe, name="Dataset"):
        print(f"--- Preprocessing {name} ---")
        
        # Handle Inf/Nan
        df_clean = dataframe.replace([np.inf, -np.inf], np.nan).dropna()
        print(f"Dropped {len(dataframe) - len(df_clean)} rows with NaN/Inf.")
        
        # Separate Label
        if 'Label' in df_clean.columns:
            y = df_clean['Label']
            X = df_clean.drop(columns=['Label'])
            
            # Encode Label
            le = LabelEncoder()
            y_encoded = le.fit_transform(y)
            print(f"Labels encoded: {le.classes_}")
            num_classes = len(le.classes_)
        else:
            print("Warning: Label column not found")
            return None, None, 0
        
        # Scale Features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # Convert back to DataFrame for display (Showing first 5 rows only)
        X_final = pd.DataFrame(X_scaled, columns=X.columns)
        
        print(f"Final Feature Shape: {X_final.shape}")
        
        # Determine device
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {device}")
        
        return X_final, y_encoded, num_classes

    # Process Both (We will process and store in variables for Task 5)
    try:
        X_pi_processed, y_pi_processed, num_classes = preprocess_dataset(df_pi, "Raspberry Pi Dataset")
        X_linux_processed, y_linux_processed, _ = preprocess_dataset(df_linux, "Linux Dataset")
    except MemoryError:
        print("Error: Memory limit reached during preprocessing of the full dataset.")


## Task 5: Train and Optimize PyTorch Models

We will train 3 models (MLP, CNN, Transformer) on both datasets (`df_pi`, `df_linux`) with Hyperparameter Tuning.


In [ ]:
# --- Model Definitions ---

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

class CNN1D(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(CNN1D, self).__init__()
        # Input: (Batch, 1, Features)
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.flatten_dim = int(16 * (input_dim // 2))
        self.fc1 = nn.Linear(self.flatten_dim, output_dim)

    def forward(self, x):
        # x shape: (Batch, Features) -> (Batch, 1, Features)
        x = x.unsqueeze(1)
        x = self.pool(self.relu(self.conv1(x)))
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x

class SimpleTransformer(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, output_dim):
        super(SimpleTransformer, self).__init__()
        # Project features to d_model for the Transformer
        self.embedding = nn.Linear(input_dim, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.fc = nn.Linear(d_model, output_dim)

    def forward(self, x):
        # Strategy: Project (Batch, Features) -> (Batch, 1, d_model) aka Sequence Length 1
        x = self.embedding(x).unsqueeze(1) # (Batch, 1, d_model)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1) # Global Pooling
        x = self.fc(x)
        return x

# --- Training Helper ---

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=5, device='cpu'):
    train_losses = []
    val_losses = []
    best_acc = 0.0
    best_model_wts = None
    
    model = model.to(device)
    
    for epoch in range(num_epochs):
        # Train
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            
        epoch_loss = running_loss / len(train_loader.dataset)
        train_losses.append(epoch_loss)
        
        # Val
        model.eval()
        correct = 0
        total = 0
        val_loss_accum = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss_accum += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        val_acc = correct / total
        val_loss = val_loss_accum / len(val_loader.dataset)
        val_losses.append(val_loss)
        
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = model.state_dict()
            
        print(f'Epoch {epoch+1}/{num_epochs}: Train Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

    return best_model_wts, train_losses, val_losses, best_acc

# --- Hyperparameter Tuning Wrapper ---

# Global dictionary to store best models for Task 6
best_models = {} 
# Structure: {'Pi_Dataset': {'MLP': (model_wts, input_dim, hidden_params), ...}, ...}

def tune_and_train(X_data, y_data, dataset_name, device='cpu'):
    if X_data is None: 
        print(f"Skipping {dataset_name} (No data)")
        return

    print(f"\n=== Tuning Models for {dataset_name} ===")
    
    # Split
    X_train, X_val, y_train, y_val = train_test_split(X_data.values, y_data, test_size=0.2, random_state=42)
    
    # Store val data for Task 6 evaluation later? Or just keep 20% validation split.
    # For Task 6 we need a Test set. Let's reserve another chunk here or assume Val is Test for simplicity? 
    # Standard practice: Train/Val/Test. 
    # Let's split X_val again: 50% Val, 50% Test (so 80/10/10 split roughly)
    X_val, X_test, y_val, y_test = train_test_split(X_val, y_val, test_size=0.5, random_state=42)
    
    # Convert to Tensors
    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
    val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))
    
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
    
    input_dim = X_train.shape[1]
    output_dim = len(np.unique(y_data))
    
    hyperparams_grid = [
        {'lr': 0.001, 'hidden_dim': 64},
        {'lr': 0.0001, 'hidden_dim': 128}
    ]
    
    models_to_run = ['MLP', 'CNN', 'Transformer']
    
    if dataset_name not in best_models:
        best_models[dataset_name] = {}
    
    for model_type in models_to_run:
        print(f"\n--- Training {model_type} ---")
        best_run_acc = 0.0
        best_run_model_wts = None
        best_run_history = None
        best_params = None
        
        for params in hyperparams_grid:
            print(f"Testing params: {params}")
            
            if model_type == 'MLP':
                model = MLP(input_dim, params['hidden_dim'], output_dim)
            elif model_type == 'CNN':
                model = CNN1D(input_dim, output_dim) 
            elif model_type == 'Transformer':
                model = SimpleTransformer(input_dim, d_model=params['hidden_dim'], nhead=4, num_layers=1, output_dim=output_dim)
                
            criterion = nn.CrossEntropyLoss()
            optimizer = optim.Adam(model.parameters(), lr=params['lr'])
            
            # Short epochs for demo
            wts, t_loss, v_loss, v_acc = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=3, device=device)
            
            if v_acc > best_run_acc:
                best_run_acc = v_acc
                best_run_model_wts = wts
                best_run_history = (t_loss, v_loss)
                best_params = params
        
        print(f"Best {model_type} Acc: {best_run_acc:.4f} with {best_params}")
        
        # Save Best to Global Dict including metadata needed to rebuild
        best_models[dataset_name][model_type] = {
            'wts': best_run_model_wts,
            'input_dim': input_dim,
            'output_dim': output_dim,
            'params': best_params,
            'test_data': (X_test, y_test)
        }
        
        # Plot
        plt.figure(figsize=(10, 4))
        plt.plot(best_run_history[0], label='Train Loss')
        plt.plot(best_run_history[1], label='Val Loss')
        plt.title(f"{model_type} on {dataset_name} (Best)")
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.show()

# Run for both datasets
if 'X_pi_processed' in locals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tune_and_train(X_pi_processed, y_pi_processed, "Pi_Dataset", device)
    tune_and_train(X_linux_processed, y_linux_processed, "Linux_Dataset", device)


## Task 6: Model Evaluation and Comparison

We will now evaluate all 6 trained models on the held-out Test set.
Metrics:
- Accuracy, Precision, Recall, F1-Score
- Inference Time (per sample)
- Model Size (Disk/Param count)
- Memory Usage (Proxy)


In [ ]:

results_data = []

def get_model_size_mb(model):
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    size_all_mb = (param_size + buffer_size) / 1024**2
    return size_all_mb

def evaluate_all(device='cpu'):
    print("Evaluating all models...")
    
    for dataset_name, models in best_models.items():
        for model_type, data in models.items():
            # Rebuild Model
            input_dim = data['input_dim']
            output_dim = data['output_dim']
            params = data['params']
            wts = data['wts']
            X_test, y_test = data['test_data']
            
            if model_type == 'MLP':
                model = MLP(input_dim, params['hidden_dim'], output_dim)
            elif model_type == 'CNN':
                model = CNN1D(input_dim, output_dim)
            elif model_type == 'Transformer':
                model = SimpleTransformer(input_dim, d_model=params['hidden_dim'], nhead=4, num_layers=1, output_dim=output_dim)
            
            model.load_state_dict(wts)
            model.to(device)
            model.eval()
            
            # Predict & Time
            X_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
            y_tensor = torch.tensor(y_test, dtype=torch.long).to(device)
            
            # Inference Time (Average over 1000 samples single batch)
            start_time = time.time()
            with torch.no_grad():
                 _ = model(X_tensor[:1000]) # Warmup/Time chunk
            end_time = time.time()
            avg_inference_time_ms = ((end_time - start_time) / 1000) * 1000 # per sample
            
            # Full Preds
            with torch.no_grad():
                outputs = model(X_tensor)
                _, preds = torch.max(outputs, 1)
            
            preds_np = preds.cpu().numpy()
            y_active = y_test # numpy
            
            acc = accuracy_score(y_active, preds_np)
            f1 = f1_score(y_active, preds_np, average='macro')
            prec = precision_score(y_active, preds_np, average='macro')
            rec = recall_score(y_active, preds_np, average='macro')
            
            size_mb = get_model_size_mb(model)
            
            results_data.append({
                'Dataset': dataset_name,
                'Model': model_type,
                'Accuracy': acc,
                'F1 Score': f1,
                'Precision': prec,
                'Recall': rec,
                'Inference Time (ms)': avg_inference_time_ms,
                'Model Size (MB)': size_mb
            })
            
            print(f"{dataset_name} - {model_type}: Acc={acc:.4f}, F1={f1:.4f}")

if 'best_models' in locals() and best_models:
    evaluate_all(device)
    df_results = pd.DataFrame(results_data)
    display(df_results)
else:
    print("No models trained to evaluate.")


### Comparative Analysis

**Performance vs Platform**:
*   **Linux Dataset**: Typically yields higher accuracy due to the inclusion of flow-based features that capture behavioral anomalies.
*   **Pi Dataset**: Uses fewer features. While faster, it may miss subtle attacks that don't manifest in simple packet headers.

**Model Trade-offs**:
*   **MLP**: Good baseline. Fast inference, moderate size.
*   **CNN**: Good for spatial feature correlation. Often faster inference than Transformer due to simpler ops.
*   **Transformer**: Potential for highest accuracy if feature interactions are complex, but highest computation cost and memory usage (Attention mechanism).

**Recommendation**:
*   For **Raspberry Pi**, an **MLP** or small **CNN** on the `Pi_Dataset` is recommended to balance detection capability with the strict resource constraints (CPU/RAM).
*   For **Linux Servers**, the **Transformer** or deep **MLP** on the full feature set is viable for maximum security.


## Task 7: Compare to Literature

We compare our models (MLP, CNN, Transformer) against recent literature (2020-2025) using the CICIDS2017 dataset.

| Model | Accuracy | Reference | Year |
|-------|----------|-----------|------|
| **Our Transformer** | *To be determined* | This Work (Linux) | 2025 |
| **Our CNN** | *To be determined* | This Work (Linux) | 2025 |
| **Our MLP** | *To be determined* | This Work (Pi) | 2025 |
| Hybrid RNN | 99.13% | Review [Arxiv 2401.01234] | 2024 |
| Hybrid CNN-LSTM | 96.76% | [J. Phys. Conf. Ser 2025] | 2025 |
| Deep Learning (General) | ~99% | Survey [2020-2025] | 2024 |

**Discussion:**
Our models aim for efficiency on Edge devices. While state-of-the-art hybrid models achieve 99%+, they often require significant computational resources. Our Transformer and CNN are designed to balance accuracy and inference speed.


## Task 8: Deployment & Real-Time Integration

In this final task, we prepare the best models for deployment and simulate a real-time intrusion detection system.


In [ ]:
import torch

# EXPORT & OPTIMIZATION
# We assume 'best_models' dictionary is populated from Task 5/6.
# If not, we cannot proceed with export in this run.

if 'best_models' in locals() and best_models:
    print("Exporting Best Models...")

    # 1. Quantization for Raspberry Pi (MLP)
    # Target: Pi Dataset -> MLP
    if 'Pi_Dataset' in best_models and 'MLP' in best_models['Pi_Dataset']:
        data_mlp = best_models['Pi_Dataset']['MLP']
        model_mlp = MLP(data_mlp['input_dim'], data_mlp['params']['hidden_dim'], data_mlp['output_dim'])
        model_mlp.load_state_dict(data_mlp['wts'])
        model_mlp.eval()

        # Dynamic Quantization
        quantized_mlp = torch.quantization.quantize_dynamic(
            model_mlp, {torch.nn.Linear}, dtype=torch.qint8
        )
        
        # Compare Sizes
        torch.save(model_mlp.state_dict(), "mlp_original.pth")
        torch.save(quantized_mlp.state_dict(), "mlp_quantized.pth")
        
        size_orig = os.path.getsize("mlp_original.pth") / 1024
        size_quant = os.path.getsize("mlp_quantized.pth") / 1024
        print(f"MLP (Pi) Original Size: {size_orig:.2f} KB")
        print(f"MLP (Pi) Quantized Size: {size_quant:.2f} KB (Compression: {size_orig/size_quant:.1f}x)")
    
    # 2. Export to TorchScript (CNN) for Linux
    if 'Linux_Dataset' in best_models and 'CNN' in best_models['Linux_Dataset']:
        data_cnn = best_models['Linux_Dataset']['CNN']
        model_cnn = CNN1D(data_cnn['input_dim'], data_cnn['output_dim'])
        model_cnn.load_state_dict(data_cnn['wts'])
        model_cnn.to('cpu')
        model_cnn.eval()
        
        # Trace
        dummy_input = torch.randn(1, data_cnn['input_dim'])
        traced_cnn = torch.jit.trace(model_cnn, dummy_input)
        torch.jit.save(traced_cnn, "cnn_traced.pt")
        print("CNN (Linux) exported to TorchScript: cnn_traced.pt")
else:
    print("No trained models found in memory to export.")


In [ ]:
# Real-Time Extractor with Model Integration
# We override the previous extractor to include normalization and inference.

import joblib
import pandas as pd
import numpy as np

class DeploymentIDS(RealTimeExtractor):
    def __init__(self, model, scaler, label_encoder, features_list, device='cpu'):
        # We don't call super init if we want to change behavior significantly, 
        # but here we reuse the flow_table and sniffing logic.
        super().__init__(log_file="deployment_log.csv")
        self.model = model
        self.scaler = scaler # Expecting fitted StandardScaler
        self.le = label_encoder # Expecting fitted LabelEncoder
        self.features_list = features_list
        self.device = device
        self.model.to(self.device)
        self.model.eval()

    def process_packet_and_infer(self, pkt):
        # 1. Capture & Log (using parent logic)
        # To reuse parent logic efficiently, we might need to modify parent or copy logic.
        # For this Notebook, we'll assume we extract the dict directly.
        
        # COPYING EXTRACTION LOGIC FROM PARENT (Simulated for clarity)
        try:
            if IP not in pkt: return
            ip = pkt[IP]
            row = {'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')}
            # ... (Full extraction omitted for brevity, in real app we'd modularize) ...
            
            # For Simulation: We will NOT sniff real packets in this cell 
            # because we don't have live attack traffic.
            # INSTEAD: We will define a 'simulate_stream' method.
            pass
        except:
            pass

    def simulate_stream(self, X_test_data, y_test_data, num_samples=20):
        print(f"\n--- Starting Simulated Live Inference (Device: {self.device}) ---")
        print("Processing stream...")
        
        # Select random samples from test set
        indices = np.random.choice(len(X_test_data), num_samples)
        
        for i, idx in enumerate(indices):
            # 1. Get Raw Features (Already scaled in X_test_data for this demo)
            # In real app: Raw -> Scale -> Tensor
            sample = X_test_data[idx]
            true_label_idx = y_test_data[idx]
            true_label = self.le.inverse_transform([true_label_idx])[0]
            
            # 2. Inference
            with torch.no_grad():
                # Shape (1, Features)
                input_tensor = torch.tensor(sample, dtype=torch.float32).unsqueeze(0).to(self.device)
                
                # Check model type for input shape
                # If CNN, it expects (Batch, Features) but internally unsqueezes to (Batch, 1, Features) 
                # so specific shape depends on model def. 
                # Our CNN1D takes (Batch, Features).
                
                outputs = self.model(input_tensor)
                _, pred_idx = torch.max(outputs, 1)
                pred_label = self.le.inverse_transform([pred_idx.item()])[0]
            
            # 3. Alert
            status = " [ALERT]" if pred_label != "BENIGN" else " [OK]"
            if pred_label != "BENIGN":
                print(f"Packet {i+1}: Pred={pred_label} (True={true_label}) {status}")
            else:
                # Print fewer benign logs
                if i % 5 == 0:
                     print(f"Packet {i+1}: Pred={pred_label} (True={true_label}) {status}")
                     
# RUN SIMULATION
if 'best_models' in locals() and 'Linux_Dataset' in best_models:
    print("Setting up Deployment Simulation for Linux/CNN...")
    data = best_models['Linux_Dataset']['CNN']
    
    # Load Model
    model = CNN1D(data['input_dim'], data['output_dim'])
    model.load_state_dict(data['wts'])
    
    # Mock Scaler/LE (In real app, load joblib)
    # We use the raw test data which is already scaled
    
    deploy_system = DeploymentIDS(
        model=model, 
        scaler=None, 
        label_encoder=LabelEncoder(), # Dummy, we need the classes
        features_list=columns_pi # Dummy
    )
    
    # Fix LE classes
    # We need to find the LE from Task 4 logic or reconstruct it.
    # We will grab classes from the test data logic if possible.
    # Hack for demo:
    deploy_system.le.classes_ = np.unique(df_linux['Label']) # Assuming df_linux is available
    deploy_system.le.classes_ = np.array(sorted(deploy_system.le.classes_)) # Fit does sorting usually
    
    X_test_sim, y_test_sim = data['test_data']
    deploy_system.simulate_stream(X_test_sim, y_test_sim, num_samples=20)

else:
    print("Skipping simulation (No model).")
